In [5]:
import pandas as pd
from sentence_transformers import SentenceTransformer
import numpy as np
from bertopic import BERTopic
from sklearn.linear_model import LinearRegression
import json

# Loading Embeddings

In [6]:
embeddings = np.load("embeddings.npy")

df = pd.read_parquet("documents.parquet")

# Model Decision

In [7]:
from sklearn.feature_extraction.text import CountVectorizer
from gensim.corpora import Dictionary
from gensim.models import CoherenceModel

vectorizer = CountVectorizer(stop_words="english")
analyzer = vectorizer.build_analyzer()

In [8]:
tokenized_docs  = [analyzer(doc) for doc in df['text'].to_list()]

In [9]:
def compute_coherence(topic_model, docs, sample_size=None, coherence_type="c_v"):
    """
    Faster coherence computation for BERTopic
    """

    # 🔹 Sample kullan (çok önerilir)
    if sample_size is not None:
        docs = docs[:sample_size]

    # 🔹 Tokenize
    vectorizer = CountVectorizer(stop_words="english")
    analyzer = vectorizer.build_analyzer()
    tokenized_docs = [analyzer(doc) for doc in docs]

    # 🔹 Dictionary
    dictionary = Dictionary(tokenized_docs)

    # 🔹 Topic words al
    topics = topic_model.get_topics()
    topic_words = [
        [word for word, _ in topics[topic_id]]
        for topic_id in topics
        if topic_id != -1
    ]

    # 🔹 Coherence hesapla
    coherence_model = CoherenceModel(
        topics=topic_words,
        texts=tokenized_docs,
        dictionary=dictionary,
        coherence=coherence_type
    )

    return coherence_model.get_coherence()

In [10]:
docs = df["text"].tolist()

results = []
models = {}
topics_dict = {} 
for min_size in [20, 50, 100]:

    model = BERTopic(
        min_topic_size=min_size,
        calculate_probabilities=False,
        verbose=True
    )

    topics, _ = model.fit_transform(docs, embeddings)

    score = compute_coherence(
        model,
        docs,
        sample_size=5000  # hız için sample
    )
    topic_count = len(model.get_topic_info()) - 1 
    results.append((min_size, score, topic_count))
    models[min_size] = model
    topics_dict[min_size] = topics

print(results)

2026-03-11 20:09:12,505 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 20:10:19,835 - BERTopic - Dimensionality - Completed ✓
2026-03-11 20:10:19,838 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 20:10:49,276 - BERTopic - Cluster - Completed ✓
2026-03-11 20:10:49,312 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 20:11:10,088 - BERTopic - Representation - Completed ✓
2026-03-11 20:12:21,432 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-03-11 20:12:58,114 - BERTopic - Dimensionality - Completed ✓
2026-03-11 20:12:58,116 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-03-11 20:13:07,714 - BERTopic - Cluster - Completed ✓
2026-03-11 20:13:07,736 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 20:13:29,916 - BERTopic - Representation - Completed ✓
2026-03-11 20:14:08,103 - BERTopic - D

[(20, np.float64(0.43915376008312507), 717), (50, np.float64(0.46615279504922275), 357), (100, np.float64(0.49419712114453784), 206)]


In [11]:
df_results = pd.DataFrame(results, columns=["min_topic_size", "coherence", "topic_count"])
df_results

,min_topic_size,coherence,topic_count
0,20,0.439154,717
1,50,0.466153,357
2,100,0.494197,206


In [12]:
model = models[100]
topics = topics_dict[100]
reduced_model = model.reduce_topics(docs, nr_topics=100)
reduced_topics, _ = reduced_model.transform(docs, embeddings)

2026-03-11 20:15:52,596 - BERTopic - Topic reduction - Reducing number of topics
2026-03-11 20:15:52,695 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-03-11 20:16:15,277 - BERTopic - Representation - Completed ✓
2026-03-11 20:16:15,305 - BERTopic - Topic reduction - Reduced number of topics from 207 to 100
2026-03-11 20:16:19,223 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-03-11 20:16:19,632 - BERTopic - Dimensionality - Completed ✓
2026-03-11 20:16:19,633 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-03-11 20:16:40,158 - BERTopic - Cluster - Completed ✓


In [13]:
#model = BERTopic(
#    min_topic_size=100,
#    calculate_probabilities=False,
#    verbose=True,
#    nr_topics=100)

# Trend Analysis

In [14]:
df["topic"] = reduced_topics
df["month"] = df["published"].dt.to_period("M")

trend = (
    df[df.topic != -1] # -1 means outliers, we ignore them
    .groupby(["month", "topic"])
    .size()
    .unstack(fill_value=0)
)

trend_share = trend.div(trend.sum(axis=1), axis=0)

In [15]:
def compute_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values

        if y.sum() == 0:
            continue

        model = LinearRegression()
        model.fit(x, y)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)

def compute_log_trend_scores(trend_share, window=12, min_share=0.001):
    scores = {}

    recent = trend_share.tail(window)
    x = np.arange(len(recent)).reshape(-1, 1)

    mean_share = recent.mean()
    valid_topics = mean_share[mean_share > min_share].index

    for topic in valid_topics:
        y = recent[topic].values



        y_log = np.log(y + 1e-6)

        model = LinearRegression()
        model.fit(x, y_log)

        scores[topic] = model.coef_[0]

    return pd.Series(scores).sort_values(ascending=False)



In [16]:
log_trend_scores_12 = compute_log_trend_scores(trend_share, window=12)

print("🔥 Emerging topics:")
print(log_trend_scores_12.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_12.tail(10))

🔥 Emerging topics:
84    0.274543
43    0.080681
82    0.071592
41    0.069905
15    0.064389
38    0.049149
78    0.044914
4     0.038469
79    0.036202
35    0.031485
dtype: float64

📉 Declining topics:
19   -0.076511
68   -0.249130
90   -0.262605
72   -0.277593
92   -0.284724
80   -0.289977
74   -0.292462
77   -0.318712
50   -0.322720
73   -0.407134
dtype: float64


In [17]:
log_trend_scores_24 = compute_log_trend_scores(trend_share, window=24)

print("🔥 Emerging topics:")
print(log_trend_scores_24.head(10))

print("\n📉 Declining topics:")
print(log_trend_scores_24.tail(10))

🔥 Emerging topics:
79    0.125321
15    0.066451
4     0.066419
66    0.065400
43    0.043468
56    0.026667
39    0.023494
35    0.023071
78    0.021897
71    0.020592
dtype: float64

📉 Declining topics:
19   -0.047934
92   -0.058384
72   -0.067161
90   -0.084148
74   -0.089795
77   -0.091856
73   -0.101935
68   -0.105211
50   -0.114283
89   -0.114894
dtype: float64


In [18]:
len(log_trend_scores_12), len(log_trend_scores_24)

(88, 91)

#  Visualization

In [19]:
import plotly.express as px
from sklearn.preprocessing import MinMaxScaler


In [20]:
def clean_label(name):
    parts = name.split("_")[2:]  # topic_x_ kısmını at
    return " ".join(parts[:3])   # ilk 3 keyword yeterli

def generate_label(topic_id, top_n=3):
    words = [word for word, _ in reduced_model.get_topic(topic_id)]
    return " / ".join(words[:top_n])

def assign_quadrant(row):
    if row["slope_24m"] > 0 and row["slope_12m"] > 0:
        return "🚀 Strong & Accelerating"
    
    elif row["slope_24m"] > 0 and row["slope_12m"] < 0:
        return "📈 Strong but Slowing"
    
    elif row["slope_24m"] < 0 and row["slope_12m"] > 0:
        return "🌱 Emerging"
    
    else:
        return "📉 Declining"

In [21]:
topics_df  = reduced_model.get_topic_info()
topics_df = topics_df[topics_df.Topic != -1]

In [22]:
topics_df["label"] = topics_df["Name"].apply(clean_label)
topics_df["label"] = topics_df["label"].str.capitalize()

In [23]:
slope_df =pd.DataFrame({
    "slope_12m": log_trend_scores_12,
    "slope_24m": log_trend_scores_24
})
slope_df["Topic"] = slope_df.index

topics_df = topics_df.merge(slope_df, on="Topic")
topics_df.set_index("Topic", inplace=True)


In [24]:
topic_map = topics_df[["label"]].to_dict()["label"]

In [25]:
topics_df["category"] = topics_df.apply(assign_quadrant, axis=1)
topics_df["acceleration"] = topics_df["slope_12m"] - topics_df["slope_24m"]

In [26]:
topics_df["growth_score"] = (topics_df["slope_12m"] * np.log1p(topics_df["Count"]))
scaler = MinMaxScaler()
topics_df["growth_size"] = scaler.fit_transform(topics_df[["growth_score"]])
topics_df["growth_size"] = topics_df["growth_size"] * 40 + 10

In [27]:
topics_df.head()

,Count,Name,Representation,Representative_Docs,label,slope_12m,slope_24m,category,acceleration,growth_score,growth_size
Topic,,,,,,,,,,,
0,9377,0_policy_reinforcement_learning_rl,"[policy, reinforcement, learning, rl, robot, d...",[Minimizing Safety Interference for Safe and C...,Reinforcement learning rl,-0.006144,-0.003524,📉 Declining,-0.002620,-0.056194,34.112619
1,5746,1_speech_asr_audio_recognition,"[speech, asr, audio, recognition, speaker, the...",[Controlling Whisper: Universal Acoustic Adver...,Asr audio recognition,-0.004005,-0.001974,📉 Declining,-0.002031,-0.034672,34.347677
2,5720,2_explanations_preference_models_to,"[explanations, preference, models, to, of, the...",[Towards Reliable Alignment: Uncertainty-aware...,Preference models to,-0.016090,-0.017387,📉 Declining,0.001297,-0.139207,33.205968
3,5127,3_summarization_retrieval_rag_entity,"[summarization, retrieval, rag, entity, the, a...",[Topic Modeling Based Extractive Text Summariz...,Retrieval rag entity,-0.026805,-0.025492,📉 Declining,-0.001313,-0.228981,32.225472
4,4188,4_reasoning_mathematical_cot_llms,"[reasoning, mathematical, cot, llms, models, t...",[Small Models Struggle to Learn from Strong Re...,Mathematical cot llms,0.038469,0.066419,🚀 Strong & Accelerating,-0.027950,0.320839,38.230504


In [28]:
topics_df["label"].nunique()

91

In [29]:
def get_topic_words(topic_id, top_n=10, model=reduced_model):
    
    words = model.get_topic(topic_id)
    
    words = words[:top_n]
    
    terms = [w[0] for w in words]
    scores = [w[1] for w in words]
    
    return terms, scores

In [30]:
topic_words = {}
for topic_id in topics_df.index:
    terms, scores = get_topic_words(topic_id)
    topic_words[topic_id] = (terms, scores)  

In [31]:
with open('topic_words.json', 'w') as f:
    json.dump(topic_words, f)

In [32]:
topics_df.to_csv("topics_trend_analysis.csv")
trend_share.to_csv("trend_share.csv")


In [33]:
top_topics = topics_df["slope_12m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 12-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [34]:
top_topics = topics_df["slope_24m"].sort_values(ascending=False).head(10).index

df_plot = trend_share[top_topics]
df_plot.index = df_plot.index.to_timestamp()
df_plot.columns = df_plot.columns.map(topic_map)

fig = px.line(
    df_plot,
    x=df_plot.index,
    y=df_plot.columns,
    title="Trend Share Over Time 24-Month Slope"
)
fig.update_layout(
    width=1200,
    height=600
)
fig.show()

In [35]:
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",

    title="AI-Powered Research Trend Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [36]:
top_topics = topics_df[(topics_df["slope_12m"] > 0)].sort_values(by="acceleration", ascending=False).head(10).index
topics_df = topics_df.loc[top_topics]
    
fig = px.scatter(
    topics_df,
    x="slope_24m",
    y="slope_12m",
    color="category",
    size="Count",
    hover_name="label",
    title="AI-Powered Research Trend Quadrant",
)


fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    xaxis_title="24M Growth Rate",
    yaxis_title="12M Growth Rate",
    template="plotly_white"
)

fig.show()

In [37]:
df_plot = topics_df.dropna()
df_plot["category"] = df_plot.apply(assign_quadrant, axis=1)
fig = px.scatter(
    df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [38]:
top10_growth = (topics_df.sort_values("growth_score", ascending=False).head(10).index)

topics_df_plot = topics_df.loc[top10_growth]
fig = px.scatter(
    topics_df_plot,
    x="slope_24m",
    y="slope_12m",
    size="growth_size",
    color="category",
    hover_name="label",
    title="AI-Powered Research Trend Intelligence Quadrant"
)

fig.add_vline(x=0, line_dash="dash")
fig.add_hline(y=0, line_dash="dash")

fig.update_layout(template="plotly_white")

fig.show()

In [39]:
impact_df = topics_df[(topics_df['slope_12m'] > 0)&
                    (topics_df["acceleration"] > 0)].copy()

impact_df = impact_df.sort_values('growth_score', ascending=False)

top10_impact = impact_df.head(10)

In [40]:
top10_impact.to_csv("top10_impact_topics.csv")

In [41]:
rep_docs = reduced_model.get_representative_docs()

top_docs = 5
rows = []

for topic_id, docs in rep_docs.items():
    for rant, text in enumerate(docs[:top_docs], start=1):
        rows.append({
            "topic_id": topic_id,
            "rank": rant,
            "text": text[:200]
        })
rep_docs_df = pd.DataFrame(rows)
rep_docs_df.to_csv("representative_docs.csv", index=False)

In [42]:
rep_docs[0]

['Minimizing Safety Interference for Safe and Comfortable Automated Driving with Distributional Reinforcement Learning Despite recent advances in reinforcement learning (RL), its application in safety critical domains like autonomous vehicles is still challenging. Although punishing RL agents for risky situations can help to learn safe policies, it may also lead to highly conservative behavior. In this paper, we propose a distributional RL framework in order to learn adaptive policies that can tune their level of conservativity at run-time based on the desired comfort and utility. Using a proactive safety verification approach, the proposed framework can guarantee that actions generated from RL are fail-safe according to the worst-case assumptions. Concurrently, the policy is encouraged to minimize safety interference and generate more comfortable behavior. We trained and evaluated the proposed approach and baseline policies using a high level simulator with a variety of randomized sce

In [43]:
rep_docs[0]

['Minimizing Safety Interference for Safe and Comfortable Automated Driving with Distributional Reinforcement Learning Despite recent advances in reinforcement learning (RL), its application in safety critical domains like autonomous vehicles is still challenging. Although punishing RL agents for risky situations can help to learn safe policies, it may also lead to highly conservative behavior. In this paper, we propose a distributional RL framework in order to learn adaptive policies that can tune their level of conservativity at run-time based on the desired comfort and utility. Using a proactive safety verification approach, the proposed framework can guarantee that actions generated from RL are fail-safe according to the worst-case assumptions. Concurrently, the policy is encouraged to minimize safety interference and generate more comfortable behavior. We trained and evaluated the proposed approach and baseline policies using a high level simulator with a variety of randomized sce

In [51]:
t = docs[0]

In [45]:
pd.read_csv("representative_docs.csv").head()

,topic_id,rank,text
0,-1,1,An Attention Matrix for Every Decision: Faithf...
1,-1,2,Distributed LLMs and Multimodal Large Language...
2,-1,3,Large Concept Models: Language Modeling in a S...
3,0,1,Minimizing Safety Interference for Safe and Co...
4,0,2,What Matters for Batch Online Reinforcement Le...


In [46]:
embeddings.shape

(179699, 384)

In [47]:
raw_data = pd.read_csv("arxiv_abstracts_final.csv").drop_duplicates()

In [48]:
raw_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 179700 entries, 0 to 182968
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   paper_id    179700 non-null  object
 1   title       179700 non-null  object
 2   abstract    179700 non-null  object
 3   published   179700 non-null  object
 4   categories  179700 non-null  object
dtypes: object(5)
memory usage: 8.2+ MB


In [54]:
paper_id = df[df['text'] == t]['paper_id']

In [56]:
raw_data[raw_data['paper_id'] == paper_id.values[0]]

,paper_id,title,abstract,published,categories
86279,http://arxiv.org/abs/2405.11537v3,VR-GPT: Visual Language Model for Intelligent ...,The advent of immersive Virtual Reality applic...,2024-05-19,"['cs.RO', 'cs.AI', 'cs.ET']"
